# FinSet: AI-Powered Financial Behavior Prediction\n\n## Problem Statement\nThis notebook demonstrates a research-oriented and classroom-friendly pipeline to predict financial behavior outcomes from transaction data.\n\n**Goals:**\n- Build reproducible ML workflows\n- Showcase live model training simulation\n- Produce academic outputs (metrics tables, plots, model artifacts)

In [1]:
# Core imports\nimport os\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score, precision_score, recall_score, f1_score,\n    classification_report, confusion_matrix\n)\nimport joblib\n\nwarnings.filterwarnings("ignore")\nSEED = 42\nnp.random.seed(SEED)\n\nOUT_DIR = "outputs"\nos.makedirs(OUT_DIR, exist_ok=True)\n\nplt.style.use("seaborn-v0_8-darkgrid")\nsns.set_context("talk", font_scale=0.8)\nprint("Environment ready. Output folder:", os.path.abspath(OUT_DIR))

## Data Analysis\nLoad transaction CSV and inspect quality.\n\nExpected columns (flexible):\n- `user_id`, `date`, `amount`, `category`, `type` (debit/credit)\n- optional target: `target`\n\nIf target is missing, this notebook creates a demo target based on risk heuristics.

In [2]:
DATA_PATH = "../data/transactions.csv"\n\nif not os.path.exists(DATA_PATH):\n    print(f"Dataset not found at {DATA_PATH}. Generating demo dataset for live classroom run...")\n    n = 1200\n    dates = pd.date_range("2025-01-01", periods=n, freq="D")\n    demo = pd.DataFrame({\n        "user_id": np.random.randint(1, 80, size=n),\n        "date": np.random.choice(dates, size=n),\n        "amount": np.round(np.random.gamma(shape=2.0, scale=1200, size=n), 2),\n        "category": np.random.choice(["Food", "Rent", "Utilities", "Shopping", "Transport", "EMI", "Subscription"], size=n),\n        "type": np.random.choice(["debit", "credit"], size=n, p=[0.75, 0.25])\n    })\n    os.makedirs("../data", exist_ok=True)\n    demo.to_csv(DATA_PATH, index=False)\n    print("Demo data created:", DATA_PATH)\n\ndf = pd.read_csv(DATA_PATH)\nprint("Shape:", df.shape)\ndisplay(df.head())\ndisplay(df.isna().sum())

SyntaxError: unexpected character after line continuation character (1162432201.py, line 1)

## Feature Engineering\nFeatures added:\n- Monthly spending per user\n- Category frequency per user\n- Weekend vs weekday spending flag\n- Basic anomaly indicator using z-score on amount\n- Expense/Income ratio per user\n- Transaction count per user

In [ ]:
# Ensure required columns exist\nrequired = ["user_id", "date", "amount", "category", "type"]\nmissing_required = [c for c in required if c not in df.columns]\nif missing_required:\n    raise ValueError(f"Dataset missing required columns: {missing_required}")\n\n# Parse dates\ndf["date"] = pd.to_datetime(df["date"], errors="coerce")\n\n# Missing-value handling (raw table stage)\ndf["category"] = df["category"].fillna("Unknown")\ndf["type"] = df["type"].fillna("debit")\ndf["amount"] = pd.to_numeric(df["amount"], errors="coerce")\ndf["amount"] = df["amount"].fillna(df["amount"].median())\ndf = df.dropna(subset=["date"]).copy()\n\n# Calendar features\ndf["month"] = df["date"].dt.to_period("M").astype(str)\ndf["day_of_week"] = df["date"].dt.day_name()\ndf["is_weekend"] = df["date"].dt.dayofweek.isin([5, 6]).astype(int)\n\n# Monthly spend per user (debit only)\nmonthly_spend = (\n    df[df["type"].str.lower() == "debit"]\n    .groupby(["user_id", "month"]) ["amount"]\n    .sum()\n    .reset_index(name="monthly_spend")\n)\nmonthly_spend_user = monthly_spend.groupby("user_id") ["monthly_spend"].mean().rename("monthly_spend_user")\n\n# Category frequency per user\ncat_freq = (\n    df.groupby(["user_id", "category"]).size().reset_index(name="cnt")\n)\ntop_cat_freq_user = cat_freq.groupby("user_id")["cnt"].max().rename("top_category_frequency")\n\n# Basic anomaly indicator: amount z-score > 2\namount_mean = df["amount"].mean()\namount_std = df["amount"].std() + 1e-6\ndf["anomaly_indicator"] = (((df["amount"] - amount_mean) / amount_std).abs() > 2).astype(int)\n\n# Aggregated user-level table\nagg = df.groupby("user_id").agg(\n    total_transactions=("amount", "count"),\n    avg_amount=("amount", "mean"),\n    total_amount=("amount", "sum"),\n    weekend_ratio=("is_weekend", "mean"),\n    anomaly_count=("anomaly_indicator", "sum"),\n).reset_index()\n\nincome = df[df["type"].str.lower() == "credit"].groupby("user_id")["amount"].sum().rename("total_income")\nexpense = df[df["type"].str.lower() == "debit"].groupby("user_id")["amount"].sum().rename("total_expense")\n\nfeatures = agg.merge(income, on="user_id", how="left").merge(expense, on="user_id", how="left")\nfeatures = features.merge(monthly_spend_user, on="user_id", how="left")\nfeatures = features.merge(top_cat_freq_user, on="user_id", how="left")\n\nfeatures["total_income"] = features["total_income"].fillna(0)\nfeatures["total_expense"] = features["total_expense"].fillna(0)\nfeatures["expense_income_ratio"] = features["total_expense"] / (features["total_income"] + 1e-6)\n\n# Dominant category per user (categorical feature)\ndominant_category = (\n    df.groupby(["user_id", "category"]).size()\n      .reset_index(name="cnt")\n      .sort_values(["user_id", "cnt"], ascending=[True, False])\n      .drop_duplicates("user_id")[["user_id", "category"]]\n      .rename(columns={"category": "dominant_category"})\n)\nfeatures = features.merge(dominant_category, on="user_id", how="left")\n\n# Create/attach target\nif "target" in df.columns:\n    # If target exists transaction-level, aggregate with mode per user\n    target_user = df.groupby("user_id")["target"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]).rename("target")\n    features = features.merge(target_user, on="user_id", how="left")\nelse:\n    # Heuristic demo target: 1 = risky behavior, 0 = stable\n    features["target"] = (\n        (features["expense_income_ratio"] > 0.9)\n        | (features["anomaly_count"] >= 3)\n        | (features["weekend_ratio"] > 0.45)\n    ).astype(int)\n\nfeatures = features.fillna({\n    "monthly_spend_user": features["monthly_spend_user"].median(),\n    "top_category_frequency": features["top_category_frequency"].median(),\n    "dominant_category": "Unknown"\n})\n\nprint("Feature table shape:", features.shape)\ndisplay(features.head())

## Model Training\nTrain/test split with reproducibility and preprocessing pipeline:\n- Missing-value imputation\n- Numerical scaling\n- Categorical encoding\n\nModels:\n1. RandomForestClassifier\n2. LogisticRegression

In [ ]:
target_col = "target"\ndrop_cols = ["user_id", target_col]\n\nX = features.drop(columns=drop_cols)\ny = features[target_col].astype(int)\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.20, random_state=SEED, stratify=y\n)\n\nnum_cols = X.select_dtypes(include=["number"]).columns.tolist()\ncat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()\n\nnumeric_preprocess = Pipeline(steps=[\n    ("imputer", SimpleImputer(strategy="median")),\n    ("scaler", StandardScaler())\n])\n\ncategorical_preprocess = Pipeline(steps=[\n    ("imputer", SimpleImputer(strategy="most_frequent")),\n    ("encoder", OneHotEncoder(handle_unknown="ignore"))\n])\n\npreprocessor = ColumnTransformer(\n    transformers=[\n        ("num", numeric_preprocess, num_cols),\n        ("cat", categorical_preprocess, cat_cols),\n    ]\n)\n\nprint("Train shape:", X_train.shape, "| Test shape:", X_test.shape)\nprint("Numerical columns:", num_cols)\nprint("Categorical columns:", cat_cols)

## Live Training Simulation\nIteratively train RandomForest with increasing complexity and update plots in real-time.

In [ ]:
rf_iterations = [20, 50, 100, 150, 220]\nrf_live_results = []\n\nplt.ion()\nfig, ax = plt.subplots(1, 2, figsize=(14, 5))\n\nfor i, n_trees in enumerate(rf_iterations, 1):\n    rf_model = Pipeline(steps=[\n        ("preprocess", preprocessor),\n        ("model", RandomForestClassifier(n_estimators=n_trees, random_state=SEED, n_jobs=-1, class_weight="balanced"))\n    ])\n    rf_model.fit(X_train, y_train)\n    preds = rf_model.predict(X_test)\n\n    acc = accuracy_score(y_test, preds)\n    prec = precision_score(y_test, preds, zero_division=0)\n    rec = recall_score(y_test, preds, zero_division=0)\n    f1 = f1_score(y_test, preds, zero_division=0)\n\n    rf_live_results.append({\n        "model": "RandomForest",\n        "iteration": i,\n        "complexity": n_trees,\n        "accuracy": acc,\n        "precision": prec,\n        "recall": rec,\n        "f1_score": f1,\n    })\n\n    print(f"[RF Iter {i}] n_estimators={n_trees} | Acc={acc:.4f} | Prec={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f}")\n\n    hist = pd.DataFrame(rf_live_results)\n    ax[0].cla(); ax[1].cla()\n    ax[0].plot(hist["complexity"], hist["accuracy"], marker="o", color="#4c78a8")\n    ax[1].plot(hist["complexity"], hist["f1_score"], marker="o", color="#f58518")\n    ax[0].set_title("RandomForest: Accuracy vs n_estimators")\n    ax[1].set_title("RandomForest: F1-score vs n_estimators")\n    ax[0].set_xlabel("n_estimators"); ax[1].set_xlabel("n_estimators")\n    ax[0].set_ylabel("Accuracy"); ax[1].set_ylabel("F1-score")\n    plt.tight_layout()\n    plt.pause(0.35)\n    time.sleep(0.35)\n\nplt.ioff()\nplt.show()\n\n# Best RF from iterative run\nrf_best_row = pd.DataFrame(rf_live_results).sort_values("f1_score", ascending=False).iloc[0]\nbest_rf_trees = int(rf_best_row["complexity"])\nprint("Best RF n_estimators:", best_rf_trees)

## Final Model Fitting (RandomForest + LogisticRegression)

In [ ]:
# Final RF\nrf_final = Pipeline(steps=[\n    ("preprocess", preprocessor),\n    ("model", RandomForestClassifier(\n        n_estimators=best_rf_trees,\n        random_state=SEED,\n        n_jobs=-1,\n        class_weight="balanced"\n    ))\n])\n\n# Logistic Regression\nlog_final = Pipeline(steps=[\n    ("preprocess", preprocessor),\n    ("model", LogisticRegression(\n        random_state=SEED,\n        max_iter=800,\n        class_weight="balanced"\n    ))\n])\n\nrf_final.fit(X_train, y_train)\nlog_final.fit(X_train, y_train)\n\nrf_pred = rf_final.predict(X_test)\nlog_pred = log_final.predict(X_test)

## Evaluation\nCompute Accuracy, Precision, Recall, F1-score, classification report, and confusion matrix.

In [ ]:
def eval_model(name, y_true, y_pred):\n    return {\n        "model": name,\n        "accuracy": accuracy_score(y_true, y_pred),\n        "precision": precision_score(y_true, y_pred, zero_division=0),\n        "recall": recall_score(y_true, y_pred, zero_division=0),\n        "f1_score": f1_score(y_true, y_pred, zero_division=0),\n    }\n\nrf_metrics = eval_model("RandomForest", y_test, rf_pred)\nlog_metrics = eval_model("LogisticRegression", y_test, log_pred)\n\nmetrics_df = pd.DataFrame([rf_metrics, log_metrics]).sort_values("f1_score", ascending=False)\ndisplay(metrics_df)\n\nprint("\\nClassification Report - RandomForest")\nprint(classification_report(y_test, rf_pred, zero_division=0))\n\nprint("Classification Report - LogisticRegression")\nprint(classification_report(y_test, log_pred, zero_division=0))

In [ ]:
# Confusion matrix for best model\nbest_model_name = metrics_df.iloc[0]["model"]\nbest_pred = rf_pred if best_model_name == "RandomForest" else log_pred\n\ncm = confusion_matrix(y_test, best_pred)\nplt.figure(figsize=(6, 5))\nsns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)\nplt.title(f"Confusion Matrix - {best_model_name}")\nplt.xlabel("Predicted")\nplt.ylabel("Actual")\nplt.tight_layout()\ncm_path = os.path.join(OUT_DIR, "confusion_matrix.png")\nplt.savefig(cm_path, dpi=160)\nplt.show()\nprint("Saved:", cm_path)

## Live Visualization Summary\nSave dynamic training graph and model metrics for report usage.

In [ ]:
rf_hist_df = pd.DataFrame(rf_live_results)\n\nplt.figure(figsize=(10, 5))\nplt.plot(rf_hist_df["complexity"], rf_hist_df["accuracy"], marker="o", label="RF Accuracy")\nplt.plot(rf_hist_df["complexity"], rf_hist_df["f1_score"], marker="o", label="RF F1-score")\nplt.title("Performance vs Model Complexity")\nplt.xlabel("RandomForest n_estimators")\nplt.ylabel("Score")\nplt.legend()\nplt.tight_layout()\nperf_path = os.path.join(OUT_DIR, "performance_graph.png")\nplt.savefig(perf_path, dpi=160)\nplt.show()\nprint("Saved:", perf_path)

## Research Outputs\nSave:\n- `metrics.csv`\n- `predictions.csv`\n- `model_v1.pkl`

In [ ]:
metrics_csv = os.path.join(OUT_DIR, "metrics.csv")\nmetrics_df.to_csv(metrics_csv, index=False)\n\npredictions_df = X_test.copy()\npredictions_df["y_true"] = y_test.values\npredictions_df["rf_pred"] = rf_pred\npredictions_df["log_pred"] = log_pred\npredictions_df["best_pred"] = best_pred\npred_csv = os.path.join(OUT_DIR, "predictions.csv")\npredictions_df.to_csv(pred_csv, index=False)\n\nbest_model = rf_final if best_model_name == "RandomForest" else log_final\nmodel_path = os.path.join(OUT_DIR, "model_v1.pkl")\njoblib.dump(best_model, model_path)\n\nprint("Saved files:")\nprint("-", metrics_csv)\nprint("-", pred_csv)\nprint("-", model_path)

## Model Comparison & Conclusion\n\n### How the models work\n- **RandomForest**: ensemble of decision trees; handles nonlinear interactions and feature mix well.\n- **LogisticRegression**: linear baseline; interpretable and fast for benchmark comparison.\n\n### Why this is suitable for financial prediction\n- Captures behavior from spending intensity, weekend tendency, anomalies, and category patterns.\n- Provides robust baseline + nonlinear model comparison.\n- Generates reproducible, report-ready artifacts for classroom/research demonstration.\n\n### Final selection criterion\nBest model selected by highest **F1-score**, balancing precision and recall for risk-sensitive use cases.

In [ ]:
best_row = metrics_df.iloc[0]\nprint("Best Model:", best_row["model"])\nprint(f"Accuracy : {best_row['accuracy']:.4f}")\nprint(f"Precision: {best_row['precision']:.4f}")\nprint(f"Recall   : {best_row['recall']:.4f}")\nprint(f"F1-score : {best_row['f1_score']:.4f}")\nprint("\\nNotebook run complete. Outputs available in:", os.path.abspath(OUT_DIR))